# Mistral-7B LoRA Fine-tuning for PsyQA Dataset (FIXED)
## ⚠️ This version fixes the catastrophic training issues

**Key Fixes:**
1. ✅ Uses proper Mistral-Instruct chat template: `[INST] instruction [/INST] response`
2. ✅ Masks prompts in training (only trains on responses)
3. ✅ Lower learning rate to prevent catastrophic forgetting
4. ✅ Fewer epochs with better monitoring
5. ✅ Proper generation using chat template

**Evaluation Metrics**: ROUGE-L, BLEU-4, BERTScore, BERT F1

## 1. Installation and Setup

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch datasets evaluate rouge-score nltk bert-score sacrebleu sentencepiece protobuf
!pip install -q peft bitsandbytes trl scipy

In [ ]:
# Import libraries
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Transformers and model libraries
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    logging
)
logging.set_verbosity_error()

# LoRA and quantization
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from transformers import BitsAndBytesConfig

# Evaluation metrics
from evaluate import load
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# HuggingFace Token Input (Runtime)
from getpass import getpass

HF_TOKEN = getpass("Enter your HuggingFace token: ")

# Login to HuggingFace
from huggingface_hub import login
login(token=HF_TOKEN)
print("✓ Successfully logged in to HuggingFace")

## 2. Load and Preprocess Dataset

In [ ]:
# Load PsyQA dataset
def load_psyqa_data(file_path: str, max_samples: int = None):
    """
    Load PsyQA dataset from JSON file
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if max_samples:
        data = data[:max_samples]
    
    processed_data = []
    for item in data:
        processed_item = {
            'question': item['question'],
            'description': item.get('description', ''),
            'keywords': item.get('keywords', ''),
            'reference_answer': item['answers'][0]['answer_text'] if item['answers'] else '',
            'questionID': item['questionID']
        }
        processed_data.append(processed_item)
    
    return processed_data

# Load dataset
data_path = 'PsyQA_example.json'
full_data = load_psyqa_data(data_path, max_samples=200)  # Use less data to prevent overfitting

# Split into train and eval
split_idx = int(len(full_data) * 0.9)
train_data = full_data[:split_idx]
eval_data = full_data[split_idx:]

# Use same 50 samples as baseline for final evaluation
test_data = load_psyqa_data(data_path, max_samples=50)

print(f"Training samples: {len(train_data)}")
print(f"Evaluation samples: {len(eval_data)}")
print(f"Test samples (same as baseline): {len(test_data)}")

## 3. Load Model and Tokenizer with Proper Chat Template

In [ ]:
# Model configuration
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("✓ Tokenizer loaded")

# Load model
print(f"\nLoading base model {MODEL_NAME}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    token=HF_TOKEN,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for training
base_model = prepare_model_for_kbit_training(base_model)

print("✓ Base model loaded and prepared for training")

In [ ]:
# LoRA Configuration (smaller rank to prevent overfitting)
lora_config = LoraConfig(
    r=8,  # Smaller rank to be more conservative
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],  # Target only attention layers for safety
    lora_dropout=0.1,  # Higher dropout to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"✓ LoRA applied")
print(f"Trainable: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")

## 4. Prepare Training Data with Proper Mistral Chat Format

In [ ]:
def format_instruction(question: str, description: str) -> str:
    """
    Create instruction for Mistral format
    """
    if description:
        return f"你是一位专业的心理健康顾问。请回答以下问题：\n\n问题：{question}\n描述：{description}\n\n请提供专业、有帮助的回答。"
    else:
        return f"你是一位专业的心理健康顾问。请回答以下问题：\n\n问题：{question}\n\n请提供专业、有帮助的回答。"


def create_mistral_prompt(instruction: str, response: str = None) -> str:
    """
    Create Mistral-Instruct format: [INST] instruction [/INST] response
    """
    if response:
        return f"[INST] {instruction} [/INST] {response}"
    else:
        return f"[INST] {instruction} [/INST]"


# Create training examples
train_examples = []
for item in train_data:
    instruction = format_instruction(item['question'], item['description'])
    response = item['reference_answer']
    full_text = create_mistral_prompt(instruction, response)
    train_examples.append(full_text)

eval_examples = []
for item in eval_data:
    instruction = format_instruction(item['question'], item['description'])
    response = item['reference_answer']
    full_text = create_mistral_prompt(instruction, response)
    eval_examples.append(full_text)

print("Sample training example:")
print(train_examples[0][:400] + "...")
print(f"\n✓ Created {len(train_examples)} training examples")

In [ ]:
# Custom dataset with proper label masking
class MaskedDataset(torch.utils.data.Dataset):
    """
    Dataset that masks the instruction part during training
    Only computes loss on the response tokens
    """
    def __init__(self, texts, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        
        for text in tqdm(texts, desc="Tokenizing"):
            # Find where the response starts (after [/INST])
            inst_end = text.find("[/INST]")
            if inst_end == -1:
                continue
            
            prompt_part = text[:inst_end + 7]  # Include "[/INST] "
            response_part = text[inst_end + 7:]
            
            # Tokenize full text
            full_tokens = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )
            
            # Tokenize just the prompt to know where to mask
            prompt_tokens = tokenizer(
                prompt_part,
                truncation=True,
                max_length=max_length
            )
            
            # Create labels (mask the prompt part with -100)
            labels = full_tokens["input_ids"].clone()
            prompt_len = len(prompt_tokens["input_ids"])
            labels[0, :prompt_len] = -100  # Don't compute loss on instruction
            
            # Mask padding tokens too
            labels[labels == tokenizer.pad_token_id] = -100
            
            self.examples.append({
                "input_ids": full_tokens["input_ids"][0],
                "attention_mask": full_tokens["attention_mask"][0],
                "labels": labels[0]
            })
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

# Create datasets
print("Creating training dataset with masked labels...")
train_dataset = MaskedDataset(train_examples, tokenizer)
eval_dataset = MaskedDataset(eval_examples, tokenizer)

print(f"✓ Created {len(train_dataset)} training samples")
print(f"✓ Created {len(eval_dataset)} eval samples")

## 5. Fine-tune with Conservative Settings

In [ ]:
# Training arguments - MUCH more conservative
training_args = TrainingArguments(
    output_dir="./mistral-lora-psyqa-fixed",
    num_train_epochs=2,  # Fewer epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,  # Much lower learning rate
    fp16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    warmup_ratio=0.1,  # Gradual warmup
    weight_decay=0.01,
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,  # Gradient clipping
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("✓ Trainer initialized with conservative settings")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

In [ ]:
# Start training
print("="*80)
print("Starting LoRA Fine-tuning with FIXED settings...")
print("="*80)

trainer.train()

print("\n" + "="*80)
print("✓ Training completed!")
print("="*80)

In [ ]:
# Save model
output_dir = "./mistral-lora-psyqa-final-fixed"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✓ Model saved to {output_dir}")

## 6. Test Generation Before Full Evaluation

In [ ]:
# Test generation on a few examples first
def generate_response(model, tokenizer, question: str, description: str, max_tokens: int = 256) -> str:
    """
    Generate response using proper Mistral format
    """
    instruction = format_instruction(question, description)
    prompt = create_mistral_prompt(instruction)
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response part (after [/INST])
    if "[/INST]" in full_output:
        response = full_output.split("[/INST]", 1)[1].strip()
    else:
        response = full_output.strip()
    
    return response

# Test on first 3 examples
print("\n" + "="*100)
print("TESTING GENERATION ON SAMPLE EXAMPLES")
print("="*100)

for i in range(min(3, len(test_data))):
    print(f"\n{'='*100}")
    print(f"Test {i+1}")
    print(f"{'='*100}")
    print(f"Question: {test_data[i]['question']}")
    
    response = generate_response(
        model, 
        tokenizer,
        test_data[i]['question'],
        test_data[i]['description']
    )
    
    print(f"\nGenerated Response:\n{response}")
    print(f"\nReference Answer:\n{test_data[i]['reference_answer'][:200]}...")
    print(f"\nResponse Length: {len(response)} chars")

print("\n" + "="*100)
print("If the responses look reasonable, proceed with full evaluation!")
print("="*100)

## 7. Evaluation Metrics

In [ ]:
def calculate_rouge_l(predictions: List[str], references: List[str]) -> float:
    rouge = load('rouge')
    results = rouge.compute(
        predictions=predictions,
        references=references,
        rouge_types=['rougeL']
    )
    return results['rougeL'] * 100


def calculate_bleu_4(predictions: List[str], references: List[str]) -> float:
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    
    for pred, ref in zip(predictions, references):
        pred_tokens = list(pred)
        ref_tokens = [list(ref)]
        score = sentence_bleu(
            ref_tokens,
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
        bleu_scores.append(score)
    
    return np.mean(bleu_scores) * 100


def calculate_bert_score(predictions: List[str], references: List[str]) -> Dict[str, float]:
    # Filter out empty predictions
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p.strip()]
    if not valid_pairs:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    
    valid_preds, valid_refs = zip(*valid_pairs)
    
    P, R, F1 = bert_score(
        list(valid_preds),
        list(valid_refs),
        lang='zh',
        verbose=False,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    
    return {
        'precision': P.mean().item() * 100,
        'recall': R.mean().item() * 100,
        'f1': F1.mean().item() * 100
    }


def compute_all_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    print("  Computing ROUGE-L...")
    rouge_l = calculate_rouge_l(predictions, references)
    
    print("  Computing BLEU-4...")
    bleu_4 = calculate_bleu_4(predictions, references)
    
    print("  Computing BERTScore...")
    bert_scores = calculate_bert_score(predictions, references)
    
    return {
        'ROUGE-L': rouge_l,
        'BLEU-4': bleu_4,
        'BERTScore-P': bert_scores['precision'],
        'BERTScore-R': bert_scores['recall'],
        'BERTScore-F1': bert_scores['f1']
    }

print("✓ Evaluation functions defined")

## 8. Full Evaluation on Test Set

In [ ]:
# Generate predictions on full test set
print("="*80)
print("Evaluating LoRA Fine-tuned Model on Test Set")
print("="*80)

lora_predictions = []
references = []

print("\nGenerating responses...")
for item in tqdm(test_data):
    reference = item['reference_answer']
    prediction = generate_response(model, tokenizer, item['question'], item['description'])
    
    lora_predictions.append(prediction)
    references.append(reference)

# Check for empty predictions
empty_count = sum(1 for p in lora_predictions if not p.strip())
print(f"\n✓ Generated {len(lora_predictions)} predictions")
if empty_count > 0:
    print(f"⚠ Warning: {empty_count} empty predictions")

In [ ]:
# Compute metrics
print("\nComputing metrics...")
lora_metrics = compute_all_metrics(lora_predictions, references)

print("\n" + "="*80)
print("LoRA FINE-TUNED MODEL RESULTS (FIXED)")
print("="*80)
for metric, value in lora_metrics.items():
    print(f"{metric}: {value:.2f}")
print("="*80)

## 9. Compare with Baseline

In [ ]:
# Load baseline results
baseline_file = 'baseline_evaluation_results.csv'

try:
    baseline_df = pd.read_csv(baseline_file)
    baseline_mistral = baseline_df[baseline_df['Model'] == 'Mistral-7B'].iloc[0]
    baseline_metrics = {
        'ROUGE-L': baseline_mistral['ROUGE-L'],
        'BLEU-4': baseline_mistral['BLEU-4'],
        'BERTScore-P': baseline_mistral['BERTScore-P'],
        'BERTScore-R': baseline_mistral['BERTScore-R'],
        'BERTScore-F1': baseline_mistral['BERTScore-F1']
    }
    print("✓ Loaded baseline metrics from file")
except FileNotFoundError:
    baseline_metrics = {
        'ROUGE-L': 35.0,
        'BLEU-4': 15.0,
        'BERTScore-P': 75.0,
        'BERTScore-R': 73.0,
        'BERTScore-F1': 74.0
    }
    print("⚠ Using placeholder baseline values")

# Create comparison
comparison_df = pd.DataFrame([
    {'Model': 'Mistral-7B (Baseline)', **baseline_metrics},
    {'Model': 'Mistral-7B + LoRA (FIXED)', **lora_metrics}
])

# Calculate improvements
improvements = {}
for metric in ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']:
    baseline_val = baseline_metrics[metric]
    lora_val = lora_metrics[metric]
    improvement = ((lora_val - baseline_val) / baseline_val) * 100
    improvements[metric] = improvement

print("\n" + "="*100)
print("BASELINE vs LoRA (FIXED) COMPARISON")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

print("\n" + "="*100)
print("IMPROVEMENTS OVER BASELINE")
print("="*100)
for metric, improvement in improvements.items():
    symbol = "✅" if improvement > 0 else "❌"
    print(f"{symbol} {metric}: {improvement:+.2f}%")
print("="*100)

## 10. Visualization

In [ ]:
!pip install -q matplotlib seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# Comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Mistral-7B: Baseline vs LoRA Fine-tuned (FIXED)', fontsize=18, fontweight='bold')

metrics = ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']
colors = ['#3498db', '#2ecc71']  # Blue for baseline, green for LoRA

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    values = comparison_df[metric].values
    models = comparison_df['Model'].values
    
    bars = ax.bar(models, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
    
    improvement = improvements[metric]
    color = 'green' if improvement > 0 else 'red'
    bgcolor = 'lightgreen' if improvement > 0 else 'lightcoral'
    ax.text(0.5, max(values) * 0.95, f'{improvement:+.1f}%', 
            ha='center', fontsize=13, fontweight='bold', 
            color=color, bbox=dict(boxstyle='round', facecolor=bgcolor, alpha=0.7))
    
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_ylim(0, max(values) * 1.15)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)

fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.savefig('lora_vs_baseline_comparison_fixed.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved")

## 11. Save Results

In [ ]:
# Save results
comparison_df.to_csv('lora_vs_baseline_results_fixed.csv', index=False)

detailed_results = {
    'baseline_metrics': baseline_metrics,
    'lora_metrics': lora_metrics,
    'improvements': improvements,
    'sample_predictions': [
        {
            'question': test_data[i]['question'],
            'lora_prediction': lora_predictions[i],
            'reference': references[i]
        }
        for i in range(min(10, len(test_data)))
    ]
}

with open('lora_detailed_results_fixed.json', 'w', encoding='utf-8') as f:
    json.dump(detailed_results, f, ensure_ascii=False, indent=2)

print("✓ All results saved")

## Summary

### What Was Fixed:
1. ✅ **Proper Mistral-Instruct format**: Uses `[INST] instruction [/INST] response`
2. ✅ **Masked training**: Only trains on responses, not instructions
3. ✅ **Conservative hyperparameters**:
   - Lower learning rate (5e-5 instead of 2e-4)
   - Fewer epochs (2 instead of 3)
   - Smaller LoRA rank (8 instead of 16)
   - Higher dropout (0.1 instead of 0.05)
4. ✅ **Proper generation**: Extracts response after `[/INST]`
5. ✅ **Testing before full eval**: Checks outputs before running all metrics

### Expected Results:
The model should now show **improvements** over baseline, not catastrophic drops!